# NFL Big Data Bowl 2027: Full Data Breakdown in Graphs
Combine testing → NFL performance for the 510-player rookie cohort (2023–2025 drafts).
Every graph has its own cell. Use the Table of Contents on the right to jump between sections.

1. Setup & loading
2. Data overview
3. Players
4. Combine results
5. Combine tracking
6. Career successes
7. Combine → career linkage
8. Games
9. Player-play (in-game metrics)
10. Game tracking 2023–2025
11. Combine speed vs. game speed
12. Player explorer

## 1. Setup & loading

In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (11, 6)
pd.set_option('display.max_columns', 80)

DATA_DIR = Path('/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027')
if not DATA_DIR.exists():  # fall back to wherever Kaggle mounted the competition
    DATA_DIR = list(Path('/kaggle/input').rglob('players.csv'))[0].parent
print('Data folder:', DATA_DIR)
for f in sorted(DATA_DIR.iterdir()):
    print(f'  {f.name:32s} {f.stat().st_size/1e6:9.1f} MB')

In [ ]:
players  = pd.read_csv(DATA_DIR / 'players.csv')
combine  = pd.read_csv(DATA_DIR / 'combine_results.csv')
career   = pd.read_csv(DATA_DIR / 'player_career_successes.csv')
games    = pd.read_csv(DATA_DIR / 'games.csv')
ctrack   = pd.read_csv(DATA_DIR / 'combine_tracking.csv')
pplay    = pd.read_csv(DATA_DIR / 'player_play.csv', low_memory=False)

def to_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().map({'true': True, 'false': False, '1': True, '0': False, '1.0': True, '0.0': False})

for c in ['target', 'in_motion_at_ball_snap', 'pressure_allowed', 'blitzing']:
    if c in pplay.columns:
        pplay[c] = to_bool(pplay[c])

names = players.set_index('nfl_id')['display_name'].to_dict()
pos_of = players.set_index('nfl_id')['nfl_position'].to_dict()

# Position groups make the many small positions readable on charts
POS_GROUP = {'QB':'QB','RB':'RB','FB':'RB','WR':'WR','TE':'TE','T':'OL','G':'OL','C':'OL','OT':'OL','OG':'OL','OL':'OL',
             'DE':'EDGE','OLB':'EDGE','DT':'DL','NT':'DL','DL':'DL','ILB':'LB','MLB':'LB','LB':'LB',
             'CB':'CB','SS':'S','FS':'S','S':'S','DB':'CB','K':'ST','P':'ST','LS':'ST'}
players['pos_group'] = players['nfl_position'].map(POS_GROUP).fillna('Other')
group_of = players.set_index('nfl_id')['pos_group'].to_dict()
GROUP_ORDER = [g for g in ['QB','RB','WR','TE','OL','DL','EDGE','LB','CB','S','ST','Other'] if g in set(players['pos_group'])]
def label(i):
    return f'{names.get(i, i)} ({pos_of.get(i, "?")})'

print('Loaded:', {k: v.shape for k, v in dict(players=players, combine=combine, career=career,
      games=games, combine_tracking=ctrack, player_play=pplay).items()})

## 2. Data overview
How big each table is and where values are missing.

In [ ]:
tables = dict(players=players, combine_results=combine, career=career, games=games,
              combine_tracking=ctrack, player_play=pplay)
def plot_missing(df, title):
    miss = df.isna().mean().mul(100).sort_values()
    miss = miss[miss > 0]
    if miss.empty:
        print(f'{title}: no missing values'); return
    fig, ax = plt.subplots(figsize=(10, max(3, len(miss) * 0.3)))
    miss.plot.barh(ax=ax, color='indianred')
    ax.set(title=title, xlabel='% missing'); plt.show()

### Graph 1: Rows per table

In [ ]:
sizes = pd.Series({k: len(v) for k, v in tables.items()}).sort_values()
ax = sizes.plot.barh(color='steelblue', logx=True)
for i, v in enumerate(sizes):
    ax.text(v, i, f' {v:,}', va='center', fontsize=9)
ax.set(title='Rows per table (log scale)', xlabel='rows'); plt.show()

### Graph 2: Columns per table

In [ ]:
ax = pd.Series({k: v.shape[1] for k, v in tables.items()}).sort_values().plot.barh(color='darkorange')
ax.set(title='Columns per table', xlabel='columns'); plt.show()

### Graph 3: Missing values: players

In [ ]:
plot_missing(players, 'players: % missing per column')

### Graph 4: Missing values: combine results

In [ ]:
plot_missing(combine, 'combine_results: % missing per column (drill opt-outs)')

### Graph 5: Missing values: career

In [ ]:
plot_missing(career, 'career: % missing per column')

### Graph 6: Missing values: player-play

In [ ]:
plot_missing(pplay, 'player_play: % missing per column (most stats only apply to certain positions)')

## 3. Players

In [ ]:
players['birth_date'] = pd.to_datetime(players['birth_date'], errors='coerce')
players['age_at_draft'] = (pd.to_datetime(players['draft_year'].astype(str) + '-04-25') - players['birth_date']).dt.days / 365.25
players['round_lbl'] = players['draft_round'].fillna(8).astype(int).astype(str).replace('8', 'UDFA')
ROUNDS = [str(i) for i in range(1, 8)] + ['UDFA']

### Graph 7: Players by NFL position

In [ ]:
ax = players['nfl_position'].value_counts().plot.bar(color='steelblue')
ax.set(title='Players by NFL position', ylabel='players'); plt.show()

### Graph 8: Position group by draft year

In [ ]:
ax = pd.crosstab(players['pos_group'], players['draft_year']).loc[GROUP_ORDER].plot.bar()
ax.set(title='Position group by draft year', ylabel='players'); plt.show()

### Graph 9: Draft round by year

In [ ]:
ax = pd.crosstab(players['round_lbl'], players['draft_year']).reindex(ROUNDS).fillna(0).plot.bar(stacked=True)
ax.set(title='Draft round by year (UDFA = undrafted)', xlabel='round', ylabel='players'); plt.show()

### Graph 10: Top 15 college conferences

In [ ]:
ax = players['college_conference'].value_counts().head(15).sort_values().plot.barh(color='seagreen')
ax.set(title='Top 15 college conferences', xlabel='players'); plt.show()

### Graph 11: Top 20 colleges

In [ ]:
ax = players['college_name'].value_counts().head(20).sort_values().plot.barh(color='slateblue')
ax.set(title='Top 20 colleges', xlabel='players'); plt.show()

### Graph 12: Age at draft

In [ ]:
ax = sns.histplot(players['age_at_draft'].dropna(), bins=25, color='teal')
ax.set(title='Age at draft', xlabel='years'); plt.show()

### Graph 13: Overall pick distribution

In [ ]:
ax = sns.histplot(players['draft_overall_pick'].dropna(), bins=26, color='purple')
ax.set(title='Overall pick number distribution', xlabel='overall pick'); plt.show()

## 4. Combine results
Body measurements, drill results and Next Gen Stats grades.

In [ ]:
comb = combine.merge(players[['nfl_id', 'display_name', 'nfl_position', 'pos_group', 'draft_round', 'draft_overall_pick', 'round_lbl']],
                     on='nfl_id', how='left')
comb['pos_group'] = comb['pos_group'].fillna(comb['combine_position'].map(POS_GROUP)).fillna('Other')
comb['round_lbl'] = comb['round_lbl'].fillna('UDFA')
MEASURES = ['combine_height', 'combine_weight', 'hand_size', 'arm_length', 'wing_span']
DRILLS   = ['ten_yd_split', 'forty', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'bench_reps']
NGS      = ['ngs_athleticism_score', 'ngs_college_production_score', 'ngs_final_score']
ALL_COMB = [c for c in MEASURES + DRILLS + NGS if c in comb.columns]
COMB_GROUPS = [g for g in GROUP_ORDER if g in set(comb.pos_group)]
display(comb[ALL_COMB].describe().T.round(2))

### Graph 14: Distribution of combine_height

In [ ]:
ax = sns.histplot(comb['combine_height'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: combine_height', xlabel='inches'); plt.show()

### Graph 15: Distribution of combine_weight

In [ ]:
ax = sns.histplot(comb['combine_weight'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: combine_weight', xlabel='pounds'); plt.show()

### Graph 16: Distribution of hand_size

In [ ]:
ax = sns.histplot(comb['hand_size'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: hand_size', xlabel='inches'); plt.show()

### Graph 17: Distribution of arm_length

In [ ]:
ax = sns.histplot(comb['arm_length'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: arm_length', xlabel='inches'); plt.show()

### Graph 18: Distribution of wing_span

In [ ]:
ax = sns.histplot(comb['wing_span'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: wing_span', xlabel='inches'); plt.show()

### Graph 19: Distribution of ten_yd_split

In [ ]:
ax = sns.histplot(comb['ten_yd_split'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: ten_yd_split', xlabel='seconds'); plt.show()

### Graph 20: Distribution of forty

In [ ]:
ax = sns.histplot(comb['forty'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: forty', xlabel='seconds'); plt.show()

### Graph 21: Distribution of vertical

In [ ]:
ax = sns.histplot(comb['vertical'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: vertical', xlabel='inches'); plt.show()

### Graph 22: Distribution of broad_jump

In [ ]:
ax = sns.histplot(comb['broad_jump'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: broad_jump', xlabel='inches'); plt.show()

### Graph 23: Distribution of three_cone

In [ ]:
ax = sns.histplot(comb['three_cone'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: three_cone', xlabel='seconds'); plt.show()

### Graph 24: Distribution of short_shuttle

In [ ]:
ax = sns.histplot(comb['short_shuttle'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: short_shuttle', xlabel='seconds'); plt.show()

### Graph 25: Distribution of bench_reps

In [ ]:
ax = sns.histplot(comb['bench_reps'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: bench_reps', xlabel='reps'); plt.show()

### Graph 26: Distribution of ngs_athleticism_score

In [ ]:
ax = sns.histplot(comb['ngs_athleticism_score'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: ngs_athleticism_score', xlabel='score'); plt.show()

### Graph 27: Distribution of ngs_college_production_score

In [ ]:
ax = sns.histplot(comb['ngs_college_production_score'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: ngs_college_production_score', xlabel='score'); plt.show()

### Graph 28: Distribution of ngs_final_score

In [ ]:
ax = sns.histplot(comb['ngs_final_score'].dropna(), bins=25, kde=True, color='steelblue')
ax.set(title='Distribution: ngs_final_score', xlabel='score'); plt.show()

### Graph 29: Drill opt-out rate

In [ ]:
ax = comb[DRILLS].isna().mean().mul(100).sort_values().plot.barh(color='indianred')
ax.set(title='Drill opt-out rate (% of prospects with no result)', xlabel='%'); plt.show()

### Graph 30: forty by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='forty', order=COMB_GROUPS, palette='tab10')
ax.set(title='forty by position group', xlabel=''); plt.show()

### Graph 31: ten_yd_split by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='ten_yd_split', order=COMB_GROUPS, palette='tab10')
ax.set(title='ten_yd_split by position group', xlabel=''); plt.show()

### Graph 32: vertical by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='vertical', order=COMB_GROUPS, palette='tab10')
ax.set(title='vertical by position group', xlabel=''); plt.show()

### Graph 33: broad_jump by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='broad_jump', order=COMB_GROUPS, palette='tab10')
ax.set(title='broad_jump by position group', xlabel=''); plt.show()

### Graph 34: three_cone by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='three_cone', order=COMB_GROUPS, palette='tab10')
ax.set(title='three_cone by position group', xlabel=''); plt.show()

### Graph 35: short_shuttle by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='short_shuttle', order=COMB_GROUPS, palette='tab10')
ax.set(title='short_shuttle by position group', xlabel=''); plt.show()

### Graph 36: bench_reps by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='bench_reps', order=COMB_GROUPS, palette='tab10')
ax.set(title='bench_reps by position group', xlabel=''); plt.show()

### Graph 37: combine_weight by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='combine_weight', order=COMB_GROUPS, palette='tab10')
ax.set(title='combine_weight by position group', xlabel=''); plt.show()

### Graph 38: ngs_athleticism_score by position group

In [ ]:
ax = sns.boxplot(data=comb, x='pos_group', y='ngs_athleticism_score', order=COMB_GROUPS, palette='tab10')
ax.set(title='ngs_athleticism_score by position group', xlabel=''); plt.show()

### Graph 39: Height vs weight by position

In [ ]:
ax = sns.scatterplot(data=comb, x='combine_weight', y='combine_height', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.set(title='Height vs weight by position group', xlabel='weight (lb)', ylabel='height (in)'); plt.show()

### Graph 40: Weight vs 40-yard dash

In [ ]:
ax = sns.scatterplot(data=comb, x='combine_weight', y='forty', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.set(title='Bigger players run slower: weight vs 40-yard dash', xlabel='weight (lb)', ylabel='40 time (s)'); plt.show()

### Graph 41: Correlation between combine metrics

In [ ]:
plt.figure(figsize=(13, 10))
sns.heatmap(comb[ALL_COMB].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': .7})
plt.title('Correlation between combine metrics'); plt.show()

### Graph 42: NGS athleticism vs college production

In [ ]:
ax = sns.scatterplot(data=comb, x='ngs_athleticism_score', y='ngs_college_production_score', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.set(title='NGS athleticism vs college production'); plt.show()

### Graph 43: NGS final score by draft round

In [ ]:
ax = sns.boxplot(data=comb, x='round_lbl', y='ngs_final_score', order=ROUNDS, palette='viridis')
ax.set(title='NGS final score by draft round', xlabel='draft round'); plt.show()

## 5. Combine tracking
10 Hz optical tracking of every drill attempt.

In [ ]:
display(ctrack.head())
cp = ctrack[ctrack['entity_type'].astype(str).str.upper() == 'PLAYER'].copy()
attempts = cp.groupby(['event_id', 'nfl_id', 'drill_type', 'drill_name', 'attempt'], as_index=False).agg(
    frames=('s', 'size'), max_speed=('s', 'max'), max_accel=('a', 'max'), distance=('dis', 'sum'))
attempts['duration_s'] = attempts['frames'] / 10
attempts['pos_group'] = attempts['nfl_id'].map(group_of)
DRILL_ORDER = attempts.groupby('drill_type')['max_speed'].median().sort_values().index

forty = cp[cp['drill_type'] == 'FORTY_YARD_DASH'].copy()
forty['t'] = pd.to_datetime(forty['time'], errors='coerce')
forty['t'] = (forty['t'] - forty.groupby('event_id')['t'].transform('min')).dt.total_seconds().round(1)
forty['pos_group'] = forty['nfl_id'].map(group_of)
forty_best = attempts[attempts.drill_type == 'FORTY_YARD_DASH'].groupby('nfl_id')['max_speed'].max().rename('combine_top_speed')
print(f'{len(attempts):,} drill attempts by {attempts.nfl_id.nunique()} players')

### Graph 44: Drill attempts by drill type

In [ ]:
ax = attempts['drill_type'].value_counts().sort_values().plot.barh(color='steelblue')
ax.set(title='Drill attempts by drill type', xlabel='attempts'); plt.show()

### Graph 45: Top 20 specific drills

In [ ]:
ax = attempts['drill_name'].value_counts().head(20).sort_values().plot.barh(color='darkorange')
ax.set(title='Top 20 specific drills', xlabel='attempts'); plt.show()

### Graph 46: Peak speed by drill

In [ ]:
ax = sns.boxplot(data=attempts, y='drill_type', x='max_speed', order=DRILL_ORDER, palette='crest')
ax.set(title='Peak speed per attempt by drill', xlabel='max speed (yd/s)', ylabel=''); plt.show()

### Graph 47: Peak acceleration by drill

In [ ]:
ax = sns.boxplot(data=attempts, y='drill_type', x='max_accel', order=DRILL_ORDER, palette='flare')
ax.set(title='Peak acceleration per attempt by drill', xlabel='max accel (yd/s²)', ylabel=''); plt.show()

### Graph 48: Attempt duration by drill

In [ ]:
ax = sns.boxplot(data=attempts, y='drill_type', x='duration_s', order=DRILL_ORDER, palette='mako')
ax.set(title='Attempt duration by drill', xlabel='seconds', ylabel=''); plt.show()

### Graph 49: 40-yard dash speed curve by position

In [ ]:
curve = forty[forty['t'] <= 6].groupby(['pos_group', 't'])['s'].mean().reset_index()
ax = sns.lineplot(data=curve, x='t', y='s', hue='pos_group', hue_order=[g for g in GROUP_ORDER if g in set(curve.pos_group)], linewidth=2)
ax.set(title='40-yard dash: average speed over time by position group', xlabel='seconds since start', ylabel='speed (yd/s)'); plt.show()

### Graph 50: Official 40 time vs sensor top speed

In [ ]:
cmp = comb.set_index('nfl_id').join(forty_best, how='inner')
ax = sns.scatterplot(data=cmp, x='forty', y='combine_top_speed', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.set(title='Official 40 time vs sensor top speed in the 40', xlabel='official 40 (s)', ylabel='sensor top speed (yd/s)'); plt.show()

## 6. Career successes

In [ ]:
car = career.merge(players[['nfl_id', 'display_name', 'nfl_position', 'pos_group', 'draft_year', 'round_lbl']], on='nfl_id', how='left')
car['total_snaps'] = car[['career_offensive_snaps', 'career_defensive_snaps', 'career_special_teams_snaps']].sum(axis=1)
car['honors'] = car[['ap_all_pro_1st_team', 'ap_all_pro_2nd_team', 'pro_bowl_original_ballot']].sum(axis=1)
display(car.describe().T.round(1))

### Graph 51: Distribution of total_snaps

In [ ]:
ax = sns.histplot(car['total_snaps'], bins=30, color='steelblue')
ax.set(title='Distribution: total_snaps'); plt.show()

### Graph 52: Distribution of career_games_active

In [ ]:
ax = sns.histplot(car['career_games_active'], bins=30, color='teal')
ax.set(title='Distribution: career_games_active'); plt.show()

### Graph 53: Distribution of career_games_started

In [ ]:
ax = sns.histplot(car['career_games_started'], bins=30, color='darkorange')
ax.set(title='Distribution: career_games_started'); plt.show()

### Graph 54: Games started by draft round

In [ ]:
ax = sns.boxplot(data=car, x='round_lbl', y='career_games_started', order=ROUNDS, palette='viridis')
ax.set(title='Games started by draft round', xlabel='round'); plt.show()

### Graph 55: Games started by draft class

In [ ]:
ax = sns.boxplot(data=car, x='draft_year', y='career_games_started', palette='Set2')
ax.set(title='Games started by draft class (older classes have had more time)'); plt.show()

### Graph 56: Average career snaps by position group

In [ ]:
snaps = car.groupby('pos_group')[['career_offensive_snaps', 'career_defensive_snaps', 'career_special_teams_snaps']].mean()
ax = snaps.reindex([g for g in GROUP_ORDER if g in snaps.index]).plot.bar(stacked=True)
ax.set(title='Average career snaps by position group', ylabel='snaps', xlabel=''); plt.show()

### Graph 57: Total honors earned

In [ ]:
honors = car[['ap_all_pro_1st_team', 'ap_all_pro_2nd_team', 'pro_bowl_original_ballot']].sum()
ax = honors.plot.bar(color=['gold', 'silver', 'firebrick'])
ax.set(title='Total honors earned by the cohort'); plt.xticks(rotation=0); plt.show()

### Graph 58: Most decorated players

In [ ]:
top = car.sort_values(['honors', 'career_games_started'], ascending=False).head(15)
fig, ax = plt.subplots()
ax.barh(top['display_name'] + ' (' + top['nfl_position'].astype(str) + ')', top['honors'], color='firebrick')
ax.invert_yaxis(); ax.set(title='Most decorated players (All-Pro + Pro Bowl)', xlabel='honors'); plt.show()

## 7. Combine → career linkage
Which combine numbers line up with NFL playing time?

In [ ]:
link = comb.merge(car[['nfl_id', 'career_games_started', 'total_snaps', 'honors', 'draft_year']], on='nfl_id', how='inner',
                  suffixes=('', '_c'))
link['ath_tier'] = pd.qcut(link['ngs_athleticism_score'], 4, labels=['Bottom 25%', '25-50%', '50-75%', 'Top 25%'])

### Graph 59: What correlates with games started

In [ ]:
corr = link[ALL_COMB + ['draft_overall_pick']].corrwith(link['career_games_started'], method='spearman').sort_values()
ax = corr.plot.barh(figsize=(9, 7), color=np.where(corr > 0, 'seagreen', 'indianred'))
ax.axvline(0, color='black', lw=.8)
ax.set(title='Spearman correlation with career games started', xlabel='correlation'); plt.show()

### Graph 60: Combine metrics vs games started, by position

In [ ]:
rows = {g: d[ALL_COMB].corrwith(d['career_games_started'], method='spearman')
        for g, d in link.groupby('pos_group') if len(d) >= 15}
plt.figure(figsize=(14, 6))
sns.heatmap(pd.DataFrame(rows).T.reindex([g for g in GROUP_ORDER if g in rows]), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation of combine metrics with games started, by position group'); plt.show()

### Graph 61: NGS final score vs games started

In [ ]:
ax = sns.regplot(data=link, x='ngs_final_score', y='career_games_started', scatter_kws={'alpha': .4})
ax.set(title='NGS final score vs games started'); plt.show()

### Graph 62: NGS athleticism vs games started

In [ ]:
ax = sns.regplot(data=link, x='ngs_athleticism_score', y='career_games_started', scatter_kws={'alpha': .4}, color='darkorange')
ax.set(title='NGS athleticism vs games started'); plt.show()

### Graph 63: Draft pick vs games started

In [ ]:
ax = sns.regplot(data=link, x='draft_overall_pick', y='career_games_started', scatter_kws={'alpha': .4}, color='seagreen')
ax.set(title='Draft pick vs games started'); plt.show()

### Graph 64: Games started by athleticism tier

In [ ]:
ax = sns.barplot(data=link, x='ath_tier', y='career_games_started', hue='draft_year', palette='Set2')
ax.set(title='Games started by athleticism tier and draft class', xlabel='NGS athleticism tier'); plt.show()

## 8. Games

### Graph 65: Games per season

In [ ]:
ax = pd.crosstab(games['season'], games['season_type']).plot.bar(stacked=True)
ax.set(title='Games per season', ylabel='games'); plt.xticks(rotation=0); plt.show()

### Graph 66: Games per week

In [ ]:
ax = games.groupby(['season', 'week']).size().unstack(0).plot(marker='o')
ax.set(title='Games per week', ylabel='games'); plt.show()

### Graph 67: Games per team

In [ ]:
ax = pd.concat([games['home_team_abbr'], games['visitor_team_abbr']]).value_counts().sort_values().plot.bar(color='slategray', figsize=(14, 5))
ax.set(title='Games per team (home + away)'); plt.show()

## 9. Player-play (in-game Next Gen Stats)
One row per player per play. Play-level stats use de-duplicated plays.

In [ ]:
pp = pplay.merge(games[['game_id', 'season', 'season_type', 'week']], on='game_id', how='left')
pp['pos_group'] = pp['nfl_id'].map(group_of)
plays = pp.drop_duplicates(['game_id', 'play_id'])
defn = pp.groupby('nfl_id')[['tackle', 'assist', 'sack', 'tackle_for_loss', 'caused_forced_fumble']].sum()
defn['name'] = defn.index.map(label)
rec = pp[pp['route_ran'].notna()]
print(f'{len(pp):,} player-play rows | {len(plays):,} unique plays | {pp.nfl_id.nunique()} players')

### Graph 68: Snaps per season by position group

In [ ]:
ax = pp.groupby(['season', 'pos_group']).size().unstack().reindex(columns=[g for g in GROUP_ORDER if g in set(pp.pos_group)]).plot.bar(stacked=True)
ax.set(title='Rookie-cohort snaps per season by position group', ylabel='player-plays'); plt.xticks(rotation=0); plt.show()

### Graph 69: Lined-up position

In [ ]:
ax = pp['lined_up_position'].value_counts().head(20).plot.bar(color='steelblue')
ax.set(title='Lined-up position on the snap', ylabel='player-plays'); plt.show()

### Graph 70: Most snaps

In [ ]:
top = pp['nfl_id'].value_counts().head(20)
fig, ax = plt.subplots()
ax.barh([label(i) for i in top.index], top.values, color='darkorange')
ax.invert_yaxis(); ax.set(title='Most snaps in the dataset', xlabel='snaps'); plt.show()

### Graph 71: Distinct players per season

In [ ]:
ax = pp.groupby('season')['nfl_id'].nunique().plot.bar(color='seagreen')
ax.set(title='Distinct cohort players appearing per season'); plt.xticks(rotation=0); plt.show()

### Graph 72: Offensive formation

In [ ]:
ax = plays['offense_formation'].value_counts().head(15).plot.bar(color='slateblue')
ax.set(title='Offensive formation', ylabel='plays'); plt.show()

### Graph 73: Receiver alignment

In [ ]:
ax = plays['receiver_alignment'].value_counts().head(15).plot.bar(color='slateblue')
ax.set(title='Receiver alignment', ylabel='plays'); plt.show()

### Graph 74: Pass result (C/I/S/IN/R)

In [ ]:
ax = plays['pass_result'].value_counts().head(15).plot.bar(color='slateblue')
ax.set(title='Pass result (C/I/S/IN/R)', ylabel='plays'); plt.show()

### Graph 75: Coverage shell

In [ ]:
ax = plays['team_coverage_type'].value_counts().head(15).plot.bar(color='slateblue')
ax.set(title='Coverage shell', ylabel='plays'); plt.show()

### Graph 76: Man vs zone

In [ ]:
ax = plays['team_coverage_man_zone'].value_counts().head(15).plot.bar(color='slateblue')
ax.set(title='Man vs zone', ylabel='plays'); plt.show()

### Graph 77: Down

In [ ]:
ax = plays['down'].value_counts().head(15).plot.bar(color='slateblue')
ax.set(title='Down', ylabel='plays'); plt.show()

### Graph 78: EPA per play

In [ ]:
ax = sns.histplot(plays['expected_points_added'].clip(-8, 8), bins=60, color='purple')
ax.set(title='EPA per play', xlabel='expected points added'); plt.show()

### Graph 79: Yards gained per play

In [ ]:
ax = sns.histplot(plays['yards_gained'].clip(-15, 50), bins=65, color='teal')
ax.set(title='Yards gained per play (clipped -15..50)'); plt.show()

### Graph 80: Mean EPA by formation

In [ ]:
epa_form = plays.groupby('offense_formation')['expected_points_added'].agg(['mean', 'size'])
epa_form = epa_form[epa_form['size'] >= 100]['mean'].sort_values()
ax = epa_form.plot.barh(color=np.where(epa_form > 0, 'seagreen', 'indianred'))
ax.set(title='Mean EPA by formation (≥100 plays)', xlabel='EPA'); plt.show()

### Graph 81: Routes run

In [ ]:
ax = rec['route_ran'].value_counts().plot.bar(color='steelblue')
ax.set(title='Routes run', ylabel='routes'); plt.show()

### Graph 82: Separation at throw by route

In [ ]:
order = rec.groupby('route_ran')['separation_at_pass_forward'].median().sort_values().index
ax = sns.boxplot(data=rec, x='route_ran', y='separation_at_pass_forward', order=order, showfliers=False, palette='crest')
ax.set(title='Separation at throw by route', ylabel='yards'); plt.xticks(rotation=45); plt.show()

### Graph 83: Actual vs expected YAC

In [ ]:
yac = pp.dropna(subset=['yards_after_catch', 'expected_yards_after_catch'])
ax = sns.scatterplot(data=yac.sample(min(len(yac), 5000), random_state=0), x='expected_yards_after_catch', y='yards_after_catch', alpha=.3)
lim = np.nanpercentile(yac[['expected_yards_after_catch', 'yards_after_catch']].values, 99)
ax.plot([0, lim], [0, lim], 'k--', lw=1)
ax.set(title='Actual vs expected yards after catch', xlim=(-2, lim), ylim=(-5, lim * 1.5)); plt.show()

### Graph 84: Pressure rate allowed per OL

In [ ]:
ol = pp[pp['pressure_allowed'].notna()]
olr = ol.groupby('nfl_id').agg(snaps=('pressure_allowed', 'size'), pressure_rate=('pressure_allowed', 'mean'))
olr = olr[olr.snaps >= 100]
ax = sns.histplot(olr['pressure_rate'] * 100, bins=25, color='indianred')
ax.set(title='Pressure rate allowed per OL (≥100 pass-block snaps)', xlabel='% of snaps'); plt.show()

### Graph 85: Pass-rusher get-off time

In [ ]:
ax = sns.histplot(pp['player_get_off'].dropna().clip(0, 2), bins=40, color='darkorange')
ax.set(title='Pass-rusher get-off time', xlabel='seconds'); plt.show()

### Graph 86: Time to pressure

In [ ]:
ax = sns.histplot(pp['time_to_pressure'].dropna().clip(0, 6), bins=40, color='purple')
ax.set(title='Time to pressure (defenders)', xlabel='seconds'); plt.show()

### Graph 87: Top 15 solo tacklers

In [ ]:
t = defn.nlargest(15, 'tackle')
fig, ax = plt.subplots(); ax.barh(t['name'], t['tackle'], color='steelblue'); ax.invert_yaxis()
ax.set(title='Top 15 solo tacklers', xlabel='tackles'); plt.show()

### Graph 88: Top 15 sack leaders

In [ ]:
t = defn.nlargest(15, 'sack')
fig, ax = plt.subplots(); ax.barh(t['name'], t['sack'], color='firebrick'); ax.invert_yaxis()
ax.set(title='Top 15 sack leaders', xlabel='sacks'); plt.show()

## 10. Game tracking 2023–2025
The three tracking files are ~2 GB together, so they're read in chunks (only the needed columns) and summarized per player.

In [ ]:
SEASONS = ['2023', '2024', '2025']
SAMPLE_PER_PLAYER = 500
player_stats, speed_samples, event_counts = [], [], []

for season in SEASONS:
    path = DATA_DIR / f'game_tracking_{season}.csv'
    print(f'Scanning {path.name} …')
    parts, samples = [], []
    for chunk in pd.read_csv(path, usecols=['nfl_id', 'game_id', 'play_id', 's', 'a', 'dis', 'event'], chunksize=2_000_000):
        chunk = chunk[np.isfinite(chunk['s']) & (chunk['s'] >= 0)]
        parts.append(chunk.groupby('nfl_id').agg(frames=('s', 'size'), s_max=('s', 'max'), a_max=('a', 'max'),
                                                 dist=('dis', 'sum'), s_sum=('s', 'sum')))
        samples.append(chunk[['nfl_id', 's']].sample(frac=min(1, 400_000 / max(len(chunk), 1)), random_state=0))
        event_counts.append(chunk['event'].value_counts().rename(season))
    agg = pd.concat(parts).groupby(level=0).agg(frames=('frames', 'sum'), s_max=('s_max', 'max'),
                                                a_max=('a_max', 'max'), dist=('dist', 'sum'), s_sum=('s_sum', 'sum'))
    smp = pd.concat(samples)
    smp = smp.sample(frac=1, random_state=0).groupby('nfl_id').head(SAMPLE_PER_PLAYER)
    agg['s_p95'] = smp.groupby('nfl_id')['s'].quantile(.95)
    agg['season'] = season
    player_stats.append(agg.reset_index())
    smp['season'] = season
    speed_samples.append(smp)
    print(f'  {len(agg):,} players, {int(agg.frames.sum()):,} frames')

gt = pd.concat(player_stats, ignore_index=True)
gt['mean_speed'] = gt['s_sum'] / gt['frames']
gt['pos_group'] = gt['nfl_id'].map(group_of)
speeds = pd.concat(speed_samples, ignore_index=True)
events = pd.concat(event_counts, axis=1).groupby(level=0).sum()
p95 = gt.pivot(index='nfl_id', columns='season', values='s_p95')

### Graph 89: Tracking frames per season

In [ ]:
ax = gt.groupby('season')['frames'].sum().div(1e6).plot.bar(color='steelblue')
ax.set(title='Tracking frames per season (millions)', ylabel='millions of frames'); plt.xticks(rotation=0); plt.show()

### Graph 90: In-game speed distribution

In [ ]:
fig, ax = plt.subplots()
for season, d in speeds.groupby('season'):
    sns.kdeplot(d['s'], ax=ax, label=season, clip=(0, 12))
ax.set(title='In-game speed distribution (all frames, sampled)', xlabel='yd/s'); ax.legend(); plt.show()

### Graph 91: Most common tracking events

In [ ]:
ax = events.sum(axis=1).sort_values().tail(20).plot.barh(color='darkorange')
ax.set(title='Most common tracking events', xlabel='frames'); plt.show()

### Graph 92: Player high-end speed by season

In [ ]:
fig, ax = plt.subplots()
for season, d in gt.groupby('season'):
    ax.hist(d['s_p95'].dropna(), bins=np.arange(0, 12.5, .5), density=True, histtype='step', linewidth=2, label=f'{season} (n={len(d)})')
ax.set(title='Player high-end speed by season', xlabel='Player 95th-percentile speed (yd/s)', ylabel='density')
ax.legend(); plt.show()

### Graph 93: Max in-game speed by position & season

In [ ]:
ax = sns.boxplot(data=gt, x='pos_group', y='s_max', order=[g for g in GROUP_ORDER if g in set(gt.pos_group)], hue='season')
ax.set(title='Max in-game speed by position group & season', ylabel='yd/s', xlabel=''); plt.show()

### Graph 94: Top 20 fastest in-game players

In [ ]:
fast = gt.sort_values('s_max', ascending=False).drop_duplicates('nfl_id').head(20)
fig, ax = plt.subplots()
ax.barh([f'{label(i)}, {s}' for i, s in zip(fast.nfl_id, fast.season)], fast.s_max, color='firebrick')
ax.invert_yaxis(); ax.set(title='Top 20 fastest in-game players', xlabel='max speed (yd/s)'); plt.show()

### Graph 95: Returning players: 2023 vs 2024 speed

In [ ]:
d = p95[['2023', '2024']].dropna()
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(d['2023'], d['2024'], s=18, alpha=.5)
lim = max(12, float(d.max().max()) if len(d) else 12)
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='same speed')
ax.set(title=f'2023 vs 2024 · {len(d):,} returning players', xlabel='2023 P95 speed (yd/s)', ylabel='2024 P95 speed (yd/s)')
ax.legend(); plt.show()

### Graph 96: Returning players: 2024 vs 2025 speed

In [ ]:
d = p95[['2024', '2025']].dropna()
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(d['2024'], d['2025'], s=18, alpha=.5)
lim = max(12, float(d.max().max()) if len(d) else 12)
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='same speed')
ax.set(title=f'2024 vs 2025 · {len(d):,} returning players', xlabel='2024 P95 speed (yd/s)', ylabel='2025 P95 speed (yd/s)')
ax.legend(); plt.show()

## 11. Combine speed vs. game speed
The core question of the competition: does combine athleticism show up on Sundays?

In [ ]:
game_peak = gt.groupby('nfl_id').agg(game_max_speed=('s_max', 'max'), game_p95=('s_p95', 'mean'), game_dist=('dist', 'sum'))
bridge = comb.set_index('nfl_id').join(forty_best, how='left').join(game_peak, how='inner').reset_index()
bridge = bridge.merge(car[['nfl_id', 'career_games_started']], on='nfl_id', how='left')

### Graph 97: Combine 40 time vs in-game top speed

In [ ]:
ax = sns.scatterplot(data=bridge, x='forty', y='game_max_speed', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.set(title='Combine 40 time vs in-game top speed', xlabel='40 time (s)', ylabel='game max speed (yd/s)'); plt.show()

### Graph 98: Combine sensor top speed vs game top speed

In [ ]:
ax = sns.scatterplot(data=bridge, x='combine_top_speed', y='game_max_speed', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.plot([6, 12], [6, 12], 'k--', lw=1)
ax.set(title='Combine sensor top speed vs game top speed', xlabel='combine top speed (yd/s)', ylabel='game max speed (yd/s)'); plt.show()

### Graph 99: In-game P95 speed vs games started

In [ ]:
ax = sns.scatterplot(data=bridge, x='game_p95', y='career_games_started', hue='pos_group', hue_order=COMB_GROUPS, alpha=.7)
ax.set(title='In-game P95 speed vs games started', xlabel='game P95 speed (yd/s)'); plt.show()

### Graph 100: Combine ↔ game speed ↔ career correlation

In [ ]:
cols = ['forty', 'ten_yd_split', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'ngs_athleticism_score',
        'combine_top_speed', 'game_max_speed', 'game_p95', 'career_games_started']
plt.figure(figsize=(11, 9))
sns.heatmap(bridge[cols].corr(method='spearman'), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Combine ↔ game speed ↔ career (Spearman)'); plt.show()

## 12. Player explorer
Pick any player from the dropdown. Each chart below updates on its own.

In [ ]:
import ipywidgets as widgets
player_ids = sorted(set(gt.nfl_id) & set(names), key=lambda i: names[i])
picker = widgets.Dropdown(options=[(label(i), i) for i in player_ids], description='Player',
                          layout=widgets.Layout(width='460px'))
display(picker)

### Graph 101: Selected player: in-game speed by season

In [ ]:
def speed_chart(pid):
    v = gt[gt.nfl_id == pid].set_index('season').reindex(SEASONS)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(SEASONS, v['s_p95'], marker='o', lw=2, label='P95 speed')
    ax.plot(SEASONS, v['s_max'], marker='s', lw=2, label='max speed')
    ax.set(title=f'{names.get(pid, pid)} · in-game speed by season', ylabel='yd/s'); ax.legend(); plt.show()
widgets.interactive_output(speed_chart, {'pid': picker})

### Graph 102: Selected player: combine percentiles vs position

In [ ]:
def combine_chart(pid):
    row = comb[comb.nfl_id == pid]
    if not len(row):
        print('No combine results for this player.'); return
    grp = row['pos_group'].iloc[0]
    pct = {c: (comb.loc[comb.pos_group == grp, c] <= row[c].iloc[0]).mean() * 100
           for c in DRILLS + NGS if pd.notna(row[c].iloc[0])}
    for c in ['ten_yd_split', 'forty', 'three_cone', 'short_shuttle']:  # lower time is better
        if c in pct: pct[c] = 100 - pct[c]
    s = pd.Series(pct).sort_values()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(s.index, s.values, color=np.where(s > 50, 'seagreen', 'indianred'))
    ax.set(xlim=(0, 100), title=f'{names.get(pid, pid)} · combine percentile vs other {grp}s (higher = better)'); plt.show()
widgets.interactive_output(combine_chart, {'pid': picker})

# Write-up findings
The sections above describe the data. The two sections below test ideas that teams can act on:

* **Finding 1: Sensor beats stopwatch.** The combine tracking records how a player *gets* to his time (burst, time to top speed, braking in cuts). Does that predict NFL success better than the official times?
* **Finding 2: Combine speed vs. game speed.** How much of a player's combine speed shows up on Sundays, by position, and does carrying it over matter?

## 13. Finding 1: Sensor beats stopwatch
### 13a. Build sensor metrics from the combine tracking
For every player's **best attempt**, from the 10 Hz frames:
* **40-yard dash:** top speed, peak acceleration, **time to 90% of top speed** (burst), and distance needed to reach it
* **3-cone & short shuttle:** peak **braking** (largest speed drop per second), peak **re-acceleration** out of a cut, and average speed through the drill
* **Position skill drills:** average peak speed and peak braking

In [ ]:
def attempt_profile(d):
    d = d.sort_values('time')
    s = d['s'].to_numpy(); dis = d['dis'].fillna(0).to_numpy()
    ds = np.diff(s) * 10 if len(s) > 1 else np.array([np.nan])   # yd/s² from 10 Hz frames
    out = {'top_speed': np.nanmax(s), 'peak_accel': np.nanmax(d['a']), 'mean_speed': np.nanmean(s),
           'max_brake': -np.nanmin(ds), 'max_reaccel': np.nanmax(ds)}
    moving = np.flatnonzero(s > 0.5)
    if len(moving):
        start = moving[0]; target = 0.9 * out['top_speed']
        hit = np.flatnonzero(s[start:] >= target)
        if len(hit):
            out['time_to_90'] = hit[0] / 10
            out['yards_to_90'] = dis[start:start + hit[0] + 1].sum()
    return out

cp_sorted = cp.sort_values(['event_id', 'time'])
prof = pd.DataFrame([{'nfl_id': pid, 'drill_type': dt, 'event_id': ev, **attempt_profile(d)}
                     for (pid, dt, ev), d in cp_sorted.groupby(['nfl_id', 'drill_type', 'event_id'])])

def best(drill, cols_how):
    d = prof[prof['drill_type'] == drill]
    if d.empty:
        return pd.DataFrame()
    return d.groupby('nfl_id').agg(**{f'{p}_{c}': (c, how) for c, how in cols_how.items() for p in [PREFIX[drill]]})

PREFIX = {'FORTY_YARD_DASH': 's40', 'THREE_CONE_DRILL': 'cone', 'SHORT_SHUTTLE': 'shut'}
parts = [best('FORTY_YARD_DASH', {'top_speed': 'max', 'peak_accel': 'max', 'time_to_90': 'min', 'yards_to_90': 'min'}),
         best('THREE_CONE_DRILL', {'max_brake': 'max', 'max_reaccel': 'max', 'mean_speed': 'max'}),
         best('SHORT_SHUTTLE', {'max_brake': 'max', 'max_reaccel': 'max', 'mean_speed': 'max'})]
skill = prof[prof['drill_type'].str.startswith('SKILL_DRILLS')].groupby('nfl_id').agg(
    skill_top_speed=('top_speed', 'mean'), skill_max_brake=('max_brake', 'max'))
sensor = pd.concat([p for p in parts if len(p)] + [skill], axis=1)
SENSOR = list(sensor.columns)
OFFICIAL = ['forty', 'ten_yd_split', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'bench_reps']
SIZE = ['combine_height', 'combine_weight']
print(f'Sensor metrics for {len(sensor)} players:', SENSOR)
display(sensor.describe().T.round(2))

In [ ]:
# Outcomes. Classes have had different time in the league (2023 class: 3 seasons, 2025 class: 1),
# so career totals are turned into per-season rates.
f1 = (comb.set_index('nfl_id')[OFFICIAL + SIZE + NGS + ['pos_group', 'display_name', 'nfl_position', 'draft_year', 'draft_overall_pick']]
      .join(sensor, how='inner')
      .join(career.set_index('nfl_id'), how='left'))
f1['seasons'] = (2025 - f1['draft_year'] + 1).clip(lower=1)
f1['starts_per_season'] = f1['career_games_started'] / f1['seasons']
f1['active_per_season'] = f1['career_games_active'] / f1['seasons']
f1['snaps_per_season'] = (f1['career_offensive_snaps'] + f1['career_defensive_snaps']) / f1['seasons']
f1 = f1.join(game_peak[['game_max_speed', 'game_p95']], how='left')

# Position-specific on-field stats from player_play
wrx = pp[pp['pos_group'].isin(['WR', 'TE', 'RB'])].groupby('nfl_id').agg(
    separation=('separation_at_pass_forward', 'mean'),
    yac_over_exp=('yards_after_catch', 'sum'), xyac=('expected_yards_after_catch', 'sum'), n_rec=('yards_after_catch', 'count'))
wrx['yac_over_exp'] = (wrx['yac_over_exp'] - wrx['xyac']) / wrx['n_rec'].replace(0, np.nan)
olx = pp[pp['pressure_allowed'].notna()].groupby('nfl_id').agg(pressure_rate=('pressure_allowed', 'mean'), n_pb=('pressure_allowed', 'size'))
dlx = pp.groupby('nfl_id').agg(get_off=('player_get_off', 'mean'), n_rush=('player_get_off', 'count'))
f1 = f1.join(wrx[['separation', 'yac_over_exp', 'n_rec']]).join(olx).join(dlx)
OUTCOMES = {'starts_per_season': 'NFL starts per season', 'snaps_per_season': 'Off+def snaps per season',
            'game_max_speed': 'In-game top speed', 'game_p95': 'In-game P95 speed'}
print(f'{len(f1)} players with both combine tracking and NFL outcomes')

In [ ]:
# Helpers: correlation within position (rank inside each position group, so position doesn't fake a link)
def within_pos_rank(df, col):
    return df.groupby('pos_group')[col].rank(pct=True)

def within_pos_corr(df, xcols, y):
    out = {}
    yr = within_pos_rank(df, y)
    for x in xcols:
        xr = within_pos_rank(df, x)
        m = xr.notna() & yr.notna()
        out[x] = np.corrcoef(xr[m], yr[m])[0, 1] if m.sum() > 20 else np.nan
    return pd.Series(out)

from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def cv_r2(df, feats, y, seed=0):
    d = df[df[y].notna()]
    num = [f for f in feats if f != 'pos_group']
    pre = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
        (OneHotEncoder(handle_unknown='ignore'), ['pos_group']))
    model = make_pipeline(pre, RidgeCV(alphas=np.logspace(-2, 3, 20)))
    sc = cross_val_score(model, d[num + ['pos_group']], d[y], cv=RepeatedKFold(n_splits=5, n_repeats=5, random_state=seed), scoring='r2')
    return sc.mean(), sc.std()

FEATURE_SETS = {'Position + size only': SIZE, 'Official combine times': SIZE + OFFICIAL,
                'Sensor metrics': SIZE + SENSOR, 'Official + sensor': SIZE + OFFICIAL + SENSOR}
r2 = pd.DataFrame({lab: {name: cv_r2(f1, feats + ['pos_group'], y)[0] for name, feats in FEATURE_SETS.items()}
                   for y, lab in OUTCOMES.items()}).T
display(r2.round(3))

### Graph 103: Same 40 time, different burst (two example sprints)

In [ ]:
f40 = f1.dropna(subset=['forty', 's40_time_to_90'])
best_pair, best_gap = None, -1
for _, grp in f40.groupby(f40['forty'].round(2)):
    if len(grp) >= 2:
        gap = grp['s40_time_to_90'].max() - grp['s40_time_to_90'].min()
        if gap > best_gap:
            best_gap, best_pair = gap, (grp['s40_time_to_90'].idxmin(), grp['s40_time_to_90'].idxmax())
fig, ax = plt.subplots(figsize=(11, 6))
if best_pair:
    for pid, col in zip(best_pair, ['seagreen', 'firebrick']):
        ev = prof[(prof.nfl_id == pid) & (prof.drill_type == 'FORTY_YARD_DASH')].sort_values('time_to_90').iloc[0]['event_id']
        d = forty[forty.event_id == ev].sort_values('t')
        ax.plot(d['t'], d['s'], lw=2.5, color=col,
                label=f"{label(pid)}: official 40 {f1.loc[pid, 'forty']}s, 90% of top speed in {f1.loc[pid, 's40_time_to_90']:.1f}s")
ax.set(title='Identical official 40 times hide very different acceleration', xlabel='seconds since start', ylabel='speed (yd/s)')
ax.legend(); plt.show()

### Graph 104: How much new information do the sensor metrics add?

In [ ]:
corr = f1[OFFICIAL + SENSOR].corr(method='spearman').loc[SENSOR, OFFICIAL]
plt.figure(figsize=(11, max(4, len(SENSOR) * 0.5)))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Sensor metrics vs official results (|r| near 1 = same information)'); plt.show()

### Graph 105: Which combine numbers track NFL starts? (within position)

In [ ]:
c = within_pos_corr(f1, OFFICIAL + SENSOR, 'starts_per_season').dropna()
c = c.reindex(c.abs().sort_values().index)
fig, ax = plt.subplots(figsize=(10, max(5, len(c) * 0.35)))
ax.barh(c.index, c.values, color=['darkorange' if i in SENSOR else 'steelblue' for i in c.index])
ax.axvline(0, color='black', lw=.8)
ax.set(title='Within-position correlation with NFL starts per season\n(orange = sensor metric, blue = official result)', xlabel='correlation')
plt.show()

### Graph 106: Which combine numbers track in-game speed? (within position)

In [ ]:
c = within_pos_corr(f1, OFFICIAL + SENSOR, 'game_p95').dropna()
c = c.reindex(c.abs().sort_values().index)
fig, ax = plt.subplots(figsize=(10, max(5, len(c) * 0.35)))
ax.barh(c.index, c.values, color=['darkorange' if i in SENSOR else 'steelblue' for i in c.index])
ax.axvline(0, color='black', lw=.8)
ax.set(title='Within-position correlation with in-game P95 speed\n(orange = sensor metric, blue = official result)', xlabel='correlation')
plt.show()

### Graph 107: Prediction test: official times vs sensor metrics

In [ ]:
ax = r2.plot.bar(figsize=(12, 6), color=['lightgray', 'steelblue', 'darkorange', 'seagreen'])
ax.axhline(0, color='black', lw=.8)
ax.set(title='Cross-validated R² (5-fold × 5 repeats): how well each set of combine data predicts NFL outcomes',
       ylabel='R² (higher = better prediction)', xlabel='')
plt.xticks(rotation=0); ax.legend(title='Inputs'); plt.show()

### Graph 108: Position-specific: WR/TE/RB separation

In [ ]:
d = f1[(f1['pos_group'].isin(['WR', 'TE', 'RB'])) & (f1['n_rec'] >= 10)]
c = within_pos_corr(d, OFFICIAL + SENSOR, 'separation').dropna()
c = c.reindex(c.abs().sort_values().index)
fig, ax = plt.subplots(figsize=(10, max(5, len(c) * 0.35)))
ax.barh(c.index, c.values, color=['darkorange' if i in SENSOR else 'steelblue' for i in c.index])
ax.axvline(0, color='black', lw=.8)
ax.set(title=f'Pass catchers (n={len(d)}): correlation with average separation at the throw', xlabel='within-position correlation'); plt.show()

### Graph 109: Position-specific: offensive line pressure allowed

In [ ]:
d = f1[(f1['pos_group'] == 'OL') & (f1['n_pb'] >= 100)]
c = within_pos_corr(d, OFFICIAL + SENSOR, 'pressure_rate').dropna()
c = c.reindex(c.abs().sort_values().index)
fig, ax = plt.subplots(figsize=(10, max(5, len(c) * 0.35)))
ax.barh(c.index, c.values, color=['darkorange' if i in SENSOR else 'steelblue' for i in c.index])
ax.axvline(0, color='black', lw=.8)
ax.set(title=f'Offensive linemen (n={len(d)}): correlation with pressure rate allowed (negative = fewer pressures)', xlabel='correlation'); plt.show()

### Graph 110: Position-specific: pass-rusher get-off

In [ ]:
d = f1[(f1['pos_group'].isin(['DL', 'EDGE'])) & (f1['n_rush'] >= 50)]
c = within_pos_corr(d, OFFICIAL + SENSOR, 'get_off').dropna()
c = c.reindex(c.abs().sort_values().index)
fig, ax = plt.subplots(figsize=(10, max(5, len(c) * 0.35)))
ax.barh(c.index, c.values, color=['darkorange' if i in SENSOR else 'steelblue' for i in c.index])
ax.axvline(0, color='black', lw=.8)
ax.set(title=f'DL/EDGE (n={len(d)}): correlation with get-off time (negative = quicker off the snap)', xlabel='correlation'); plt.show()

### Graph 111: Burst vs official 40 for NFL starts (scatter)

In [ ]:
d = f1.dropna(subset=['forty', 's40_time_to_90', 'starts_per_season'])
d = d.assign(forty_rank=within_pos_rank(d, 'forty'), burst_rank=within_pos_rank(d, 's40_time_to_90'))
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(d['forty_rank'], d['burst_rank'], c=d['starts_per_season'], cmap='viridis', s=40, alpha=.8)
plt.colorbar(sc, label='NFL starts per season')
ax.plot([0, 1], [0, 1], 'k--', lw=.8)
ax.set(title='Official 40 vs sensor burst (both ranked within position; lower = faster)',
       xlabel='40 time rank within position', ylabel='time-to-90%-top-speed rank within position'); plt.show()

## 14. Finding 2: Combine speed vs. game speed
Combine top speed is a straight-line sprint with no pads. Game top speed is measured in real plays. How much carries over?

In [ ]:
f2 = f1.dropna(subset=['s40_top_speed', 'game_max_speed']).copy()
f2['translation'] = f2['game_max_speed'] / f2['s40_top_speed']      # 1.0 = hits combine speed in games
# Expected game speed from combine speed + position; the residual = plays faster/slower than his combine says
from sklearn.linear_model import LinearRegression
X = pd.get_dummies(f2[['s40_top_speed', 'pos_group']], columns=['pos_group'], drop_first=True).astype(float)
lr = LinearRegression().fit(X, f2['game_max_speed'])
f2['expected_game_speed'] = lr.predict(X)
f2['speed_over_expected'] = f2['game_max_speed'] - f2['expected_game_speed']
f2_groups = [g for g in GROUP_ORDER if g in set(f2.pos_group)]
print(f'{len(f2)} players with both combine and game tracking speed')
print('Median share of combine top speed reached in games:', round(f2['translation'].median() * 100, 1), '%')

### Graph 112: Share of combine top speed reached in games, by position

In [ ]:
ax = sns.boxplot(data=f2, x='pos_group', y=f2['translation'] * 100, order=f2_groups, palette='tab10')
ax.axhline(100, color='black', ls='--', lw=1)
ax.set(title='Game top speed as % of combine 40 top speed', ylabel='% of combine top speed', xlabel=''); plt.show()

### Graph 113: Which combine measure best predicts game top speed?

In [ ]:
cands = {'Official 40': ['forty'], 'Official 10-yd split': ['ten_yd_split'], 'Sensor top speed (40)': ['s40_top_speed'],
         'Sensor burst (time to 90%)': ['s40_time_to_90'], 'Skill-drill speed': ['skill_top_speed'],
         'All sensor metrics': SENSOR, 'All official results': OFFICIAL}
res = pd.Series({k: cv_r2(f2, v + ['pos_group'], 'game_max_speed')[0] for k, v in cands.items()}).sort_values()
ax = res.plot.barh(color=['darkorange' if ('Sensor' in k or 'sensor' in k or 'Skill' in k) else 'steelblue' for k in res.index])
ax.set(title='Predicting in-game top speed (position included in every model)', xlabel='cross-validated R²'); plt.show()

### Graph 114: Combine top speed vs game top speed (who over/under-translates)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=f2, x='s40_top_speed', y='game_max_speed', hue='pos_group', hue_order=f2_groups, alpha=.6, ax=ax)
lo, hi = f2[['s40_top_speed', 'game_max_speed']].min().min(), f2[['s40_top_speed', 'game_max_speed']].max().max()
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='game = combine')
for pid in list(f2['speed_over_expected'].nlargest(4).index) + list(f2['speed_over_expected'].nsmallest(4).index):
    ax.annotate(names.get(pid, pid), (f2.loc[pid, 's40_top_speed'], f2.loc[pid, 'game_max_speed']), fontsize=8)
ax.set(title='Combine 40 top speed vs in-game top speed', xlabel='combine top speed (yd/s)', ylabel='game top speed (yd/s)')
ax.legend(fontsize=8); plt.show()

### Graph 115: Biggest over- and under-translators

In [ ]:
top = pd.concat([f2['speed_over_expected'].nlargest(10), f2['speed_over_expected'].nsmallest(10)]).sort_values()
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh([label(i) for i in top.index], top.values, color=np.where(top > 0, 'seagreen', 'indianred'))
ax.axvline(0, color='black', lw=.8)
ax.set(title='Game top speed vs what his combine speed predicted (yd/s)', xlabel='yd/s faster (+) or slower (−) than expected'); plt.show()

### Graph 116: Does playing faster than your combine earn playing time?

In [ ]:
f2['translation_group'] = pd.qcut(f2['speed_over_expected'], 4, labels=['Slowest 25%', '25-50%', '50-75%', 'Fastest 25%'])
ax = sns.barplot(data=f2, x='translation_group', y='starts_per_season', palette='RdYlGn', errorbar=('ci', 90))
ax.set(title='NFL starts per season by game speed vs combine expectation', xlabel='game speed relative to combine expectation',
       ylabel='starts per season'); plt.show()

### Graph 117: Does game speed hold up from season to season?

In [ ]:
seq = gt.pivot(index='nfl_id', columns='season', values='s_max')
rows = []
for a, b in [('2023', '2024'), ('2024', '2025')]:
    d = seq[[a, b]].dropna()
    rows.append({'seasons': f'{a}→{b}', 'players': len(d), 'avg change (yd/s)': (d[b] - d[a]).mean(),
                 'correlation': d[a].corr(d[b])})
chg = pd.DataFrame(rows).set_index('seasons'); display(chg.round(3))
ax = chg['avg change (yd/s)'].plot.bar(color=np.where(chg['avg change (yd/s)'] > 0, 'seagreen', 'indianred'))
ax.axhline(0, color='black', lw=.8)
ax.set(title='Average change in a player’s in-game top speed, year over year', ylabel='yd/s'); plt.xticks(rotation=0); plt.show()

## 15. Key numbers for the write-up

In [ ]:
summary = {}
summary['Players analysed (combine tracking + NFL)'] = len(f1)
for y, lab in OUTCOMES.items():
    summary[f'R² official → {lab}'] = round(r2.loc[lab, 'Official combine times'], 3)
    summary[f'R² sensor → {lab}'] = round(r2.loc[lab, 'Sensor metrics'], 3)
    summary[f'R² both → {lab}'] = round(r2.loc[lab, 'Official + sensor'], 3)
cs = within_pos_corr(f1, OFFICIAL + SENSOR, 'starts_per_season').dropna()
summary['Strongest single link to starts'] = f'{cs.abs().idxmax()} (r = {cs[cs.abs().idxmax()]:.2f})'
summary['Official 40 link to starts'] = round(cs.get('forty', np.nan), 3)
summary['Median % of combine top speed reached in games'] = round(f2['translation'].median() * 100, 1)
summary['Game-speed translation by position (median %)'] = (f2.groupby('pos_group')['translation'].median() * 100).round(1).to_dict()
summary['Starts/season: fastest-25% translators vs slowest-25%'] = (
    f2.groupby('translation_group')['starts_per_season'].mean().round(2).to_dict())
for k, v in summary.items():
    print(f'{k}: {v}')